In [1]:
!pip install watchdog

In [2]:
import os
import math
import time
import csv
import threading
from collections import Counter
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

# Colab Configuration
WATCH_DIRECTORY = "/content/All_Documents"
LOG_FILE = "/content/file_entropy_stream.csv"

csv_lock = threading.Lock()

def calculate_shannon_entropy(file_path):
    try:
        with open(file_path, 'rb') as f:
            data = f.read()
        if not data:
            return 0.0

        file_length = len(data)
        byte_counts = Counter(data)

        entropy = 0.0
        for count in byte_counts.values():
            p_x = count / file_length
            entropy -= p_x * math.log2(p_x)

        return entropy
    except Exception:
        return -1.0

def log_event_to_csv(timestamp, file_path, file_size, entropy_score):
    with csv_lock:
        file_exists = os.path.isfile(LOG_FILE)
        with open(LOG_FILE, 'a', newline='') as csvfile:
            writer = csv.writer(csvfile)
            if not file_exists:
                writer.writerow(['Timestamp', 'File_Path', 'File_Size_Bytes', 'Entropy_Score'])
            writer.writerow([timestamp, file_path, file_size, f"{entropy_score:.4f}"])

class EntropyMonitorHandler(FileSystemEventHandler):
    def process_event(self, event):
        if event.is_directory or os.path.basename(event.src_path) == os.path.basename(LOG_FILE):
            return

        file_path = event.src_path
        time.sleep(0.05) # Give Colab filesystem time to release lock

        try:
            file_size = os.path.getsize(file_path)
            entropy_score = calculate_shannon_entropy(file_path)

            if entropy_score != -1.0:
                timestamp = time.time()
                log_event_to_csv(timestamp, file_path, file_size, entropy_score)
                # Note: Print statements from background threads don't always render clearly in Colab cells
                # We will rely on the CSV file to verify our data.
        except Exception:
            pass

    def on_modified(self, event):
        self.process_event(event)

    def on_created(self, event):
        self.process_event(event)

# Setup directory
if not os.path.exists(WATCH_DIRECTORY):
    os.makedirs(WATCH_DIRECTORY)

# Start background observer
event_handler = EntropyMonitorHandler()
observer = Observer()
observer.schedule(event_handler, WATCH_DIRECTORY, recursive=True)
observer.start()

print(f"[*] EntropyShield Phase 1 Active in Background.")
print(f"[*] Monitoring: {WATCH_DIRECTORY}")

[*] EntropyShield Phase 1 Active in Background.
[*] Monitoring: /content/All_Documents


In [3]:
import os

print("[+] Writing normal text file...")
with open(os.path.join(WATCH_DIRECTORY, "normal_doc.txt"), "w") as f:
    # Highly predictable, low entropy data
    f.write("This is a normal document. " * 100)

time.sleep(1)

print("[+] Writing simulated encrypted file...")
with open(os.path.join(WATCH_DIRECTORY, "encrypted_payload.bin"), "wb") as f:
    # Random bytes simulate the high entropy of an encrypted file
    f.write(os.urandom(2048))

print("[+] Done. Check the CSV log for results.")

[+] Writing normal text file...
[+] Writing simulated encrypted file...
[+] Done. Check the CSV log for results.


In [4]:
import pandas as pd

try:
    df = pd.read_csv(LOG_FILE)
    display(df)
except FileNotFoundError:
    print("CSV not found yet. Try running the simulation cell again or wait a second.")


,Timestamp,File_Path,File_Size_Bytes,Entropy_Score
0,1.779247e+09,/content/All_Documents/normal_doc.txt,2700,3.8805
1,1.779247e+09,/content/All_Documents/normal_doc.txt,2700,3.8805
2,1.779247e+09,/content/All_Documents/encrypted_payload.bin,2048,7.9100
3,1.779247e+09,/content/All_Documents/encrypted_payload.bin,2048,7.9100


In [5]:
import pandas as pd
import numpy as np
import os
import time

# Configuration - reading from the stream we built in Phase 1
LOG_FILE = "/content/file_entropy_stream.csv"

def process_behavioral_pipeline(window_seconds=3):
    """
    Reads the raw file tracking data and groups it into
    time-chunks to look for suspicious speed and structural shifts.
    """
    print(f"[*] Reading data from tracker: {LOG_FILE}")

    if not os.path.exists(LOG_FILE):
        print("[!] Error: No log file found yet. Please run your Phase 1 simulation cell first!")
        return

    # Load the CSV file into a pandas data table
    df = pd.read_csv(LOG_FILE)

    if df.empty or len(df) < 2:
        print("[!] Not enough file data collected yet to build a timeline pattern.")
        return

    # Convert the raw timestamps into an understandable date-time format
    df['Datetime'] = pd.to_datetime(df['Timestamp'], unit='s')

    # Set the time as the index of our table so we can slice it by time chunks
    df.set_index('Datetime', inplace=True)

    print(f"[*] Successfully loaded {len(df)} file events.")
    print(f"[*] Aggregating data into {window_seconds}-second blocks...")

    # This is the magic command. It groups rows together into blocks of time (e.g., 3-second blocks)
    # '3S' means 3 seconds.
    window_rule = f"{window_seconds}s"

    # Calculate features for each time block
    aggregated_features = df.resample(window_rule).agg(
        File_Modification_Velocity=('File_Path', 'count'),       # How many files changed in this window
        Average_Entropy=('Entropy_Score', 'mean'),               # The average randomness score
        Total_Bytes_Written=('File_Size_Bytes', 'sum'),          # Total volume of data changed
        Max_Entropy_Seen=('Entropy_Score', 'max')                # Highest single entropy score seen
    )

    # Fill any empty time gaps with zeros so our timeline doesn't break
    aggregated_features.fillna(0, inplace=True)

    # Clean up the output table format
    aggregated_features.reset_index(inplace=True)

    print("\n[+] --- BEHAVIORAL FEATURE PIPELINE OUTPUT ---")
    print("This table shows what the Machine Learning model will see in Phase 3:\n")

    # Display the final structured timeline data
    display(aggregated_features)

    return aggregated_features

# Run the pipeline with a 2-second tracking window
features_dataframe = process_behavioral_pipeline(window_seconds=2)

[*] Reading data from tracker: /content/file_entropy_stream.csv
[*] Successfully loaded 4 file events.
[*] Aggregating data into 2-second blocks...

[+] --- BEHAVIORAL FEATURE PIPELINE OUTPUT ---
This table shows what the Machine Learning model will see in Phase 3:



,Datetime,File_Modification_Velocity,Average_Entropy,Total_Bytes_Written,Max_Entropy_Seen
0,2026-05-20 03:18:42,2,3.8805,5400,3.8805
1,2026-05-20 03:18:44,2,7.9100,4096,7.9100


In [6]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print("[*] Generating synthetic training data for the AI...")

# 1. CREATE "NORMAL" BEHAVIOR DATA
# Low file modification speed (1 to 5 files per window)
# Low to medium entropy (3.0 to 6.5)
normal_data = pd.DataFrame({
    'File_Modification_Velocity': np.random.randint(1, 6, 500),
    'Average_Entropy': np.random.uniform(3.0, 6.5, 500),
    'Label': 0 # 0 means "Safe"
})

# 2. CREATE "RANSOMWARE" BEHAVIOR DATA
# High file modification speed (10 to 50 files per window)
# High entropy approaching the mathematical maximum (7.5 to 8.0)
malicious_data = pd.DataFrame({
    'File_Modification_Velocity': np.random.randint(10, 50, 500),
    'Average_Entropy': np.random.uniform(7.5, 8.0, 500),
    'Label': 1 # 1 means "Ransomware!"
})

# Combine them into one big textbook for the AI to study
training_dataset = pd.concat([normal_data, malicious_data])

# Separate the "Features" (the clues) from the "Labels" (the answers)
X = training_dataset[['File_Modification_Velocity', 'Average_Entropy']]
y = training_dataset['Label']

print("[*] Training the Scikit-Learn Random Forest Classifier...")

# Initialize the AI Model
# A Random Forest works by creating 100 mini "decision trees" and having them vote
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model (This is where the AI actually learns!)
classifier.fit(X, y)

# Let's test it on the data to see how well it learned
predictions = classifier.predict(X)
accuracy = accuracy_score(y, predictions)

print(f"[+] AI Model Training Complete! Accuracy: {accuracy * 100:.2f}%\n")

# 3. SAVE THE AI BRAIN TO A FILE
MODEL_FILENAME = "/content/entropyshield_model.p"
with open(MODEL_FILENAME, 'wb') as file:
    pickle.dump(classifier, file)

print(f"[+] Saved the trained AI brain to: {MODEL_FILENAME}")
print("\n--- QUICK AI TEST ---")

# Let's give it a fake scenario it has never seen before to test it
test_scenario = pd.DataFrame({
    'File_Modification_Velocity': [25], # 25 files modified...
    'Average_Entropy': [7.95]           # ...with near max entropy!
})

prediction = classifier.predict(test_scenario)

if prediction[0] == 1:
    print("AI Diagnosis for test scenario: 🚨 RANSOMWARE DETECTED 🚨")
else:
    print("AI Diagnosis for test scenario: ✅ Normal Behavior")

[*] Generating synthetic training data for the AI...
[*] Training the Scikit-Learn Random Forest Classifier...
[+] AI Model Training Complete! Accuracy: 100.00%

[+] Saved the trained AI brain to: /content/entropyshield_model.p

--- QUICK AI TEST ---
AI Diagnosis for test scenario: 🚨 RANSOMWARE DETECTED 🚨


In [7]:
import pandas as pd
import time
import os
import pickle
from IPython.display import clear_output, display, HTML

# Configuration
LOG_FILE = "/content/file_entropy_stream.csv"
MODEL_FILENAME = "/content/entropyshield_model.p"
WATCH_DIRECTORY = "/content/All_Documents"

def simulate_ransomware_attack():
    """Simulates an attack by rapidly creating high-entropy files."""
    print("\n[💀] MALICIOUS ACTOR: Initiating encryption payload...")
    for i in range(15):
        # Create 15 encrypted files rapidly
        with open(os.path.join(WATCH_DIRECTORY, f"locked_file_{i}.enc"), "wb") as f:
            f.write(os.urandom(4096)) # Write random (high entropy) bytes
        time.sleep(0.1) # Fast, but realistic file I/O speed

def start_guardian_daemon(loops=5):
    """
    The active defensive daemon. It checks the system behavior every 3 seconds.
    """
    print("[*] Loading AI Brain...")
    if not os.path.exists(MODEL_FILENAME):
        print("[!] Error: AI Model not found. Run Phase 3 first!")
        return

    with open(MODEL_FILENAME, 'rb') as file:
        ai_brain = pickle.load(file)

    print("[*] EntropyShield Daemon ACTIVATED. Monitoring system...\n")

    for _ in range(loops):
        time.sleep(3) # Wait 3 seconds to collect a "window" of behavior

        try:
            # 1. Read the most recent file activity
            df = pd.read_csv(LOG_FILE)

            # Grab only the last 3 seconds of data
            current_time = time.time()
            recent_activity = df[df['Timestamp'] >= (current_time - 3.0)]

            if recent_activity.empty:
                print("[✓] SYSTEM SECURE: No recent file modifications detected.")
                continue

            # 2. Calculate the features for this time window (Phase 2 logic)
            files_modified = len(recent_activity)
            avg_entropy = recent_activity['Entropy_Score'].mean()

            # 3. Feed the features to the AI (Phase 3 logic)
            current_behavior = pd.DataFrame({
                'File_Modification_Velocity': [files_modified],
                'Average_Entropy': [avg_entropy]
            })

            prediction = ai_brain.predict(current_behavior)

            # 4. Take Action!
            if prediction[0] == 1:
                # Trigger the OS Warning
                display(HTML("<h1 style='color:red; background-color:black; padding:10px;'>🚨 CRITICAL ALERT: RANSOMWARE ENCRYPTION DETECTED 🚨</h1>"))
                print(f"[!] THREAT METRICS: {files_modified} files modified in 3s | Average Entropy: {avg_entropy:.4f}")
                print("[!] ACTION TAKEN: Alerting System Administrator and suspending I/O processes...")
                break # Stop the daemon, the system is locked down
            else:
                print(f"[✓] SYSTEM SECURE: {files_modified} files modified. Normal entropy ({avg_entropy:.2f}).")

        except Exception as e:
            print(f"Daemon encountered an error: {e}")

# Run the simulation and the daemon together
import threading

# Start the attack in the background
attack_thread = threading.Thread(target=simulate_ransomware_attack)
attack_thread.start()

# Start the defense daemon in the foreground
start_guardian_daemon(loops=6)


[💀] MALICIOUS ACTOR: Initiating encryption payload...
[*] Loading AI Brain...
[*] EntropyShield Daemon ACTIVATED. Monitoring system...



[!] THREAT METRICS: 30 files modified in 3s | Average Entropy: 7.9558
[!] ACTION TAKEN: Alerting System Administrator and suspending I/O processes...


In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle

print("[*] Upgrading AI Brain with Advanced Kernel Features...")

# 1. CREATE "NORMAL" BEHAVIOR DATA (The tricky stuff!)
# Notice that normal users CAN have high speed and high entropy now (like zipping a file),
# but they are usually using a Known Process, and the File Headers are usually safe.
normal_data = pd.DataFrame({
    'File_Velocity': np.random.randint(1, 40, 500),        # Speeds can be high now
    'Average_Entropy': np.random.uniform(3.0, 8.0, 500),   # Entropy can hit 8.0 now
    'Is_Known_Process': np.random.choice([1, 1, 1, 0], 500), # 1 = Yes (Trusted Program)
    'Header_Altered': np.random.choice([0, 0, 0, 1], 500),   # 0 = Safe (Header intact)
    'Label': 0 # 0 means "Safe"
})

# 2. CREATE "RANSOMWARE" BEHAVIOR DATA
# High speed, max entropy, usually an Unknown Process, and destroys headers.
malicious_data = pd.DataFrame({
    'File_Velocity': np.random.randint(20, 80, 500),
    'Average_Entropy': np.random.uniform(7.6, 8.0, 500),
    'Is_Known_Process': np.random.choice([0, 0, 0, 1], 500), # 0 = No (Unknown Program)
    'Header_Altered': np.random.choice([1, 1, 1, 0], 500),   # 1 = Yes (Header destroyed)
    'Label': 1 # 1 means "Ransomware!"
})

# Combine and prepare the data
training_dataset = pd.concat([normal_data, malicious_data])
X = training_dataset[['File_Velocity', 'Average_Entropy', 'Is_Known_Process', 'Header_Altered']]
y = training_dataset['Label']

print("[*] Training the Advanced Scikit-Learn Classifier...")
advanced_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
advanced_classifier.fit(X, y)

print(f"[+] Advanced AI Model Training Complete! Accuracy: {accuracy_score(y, advanced_classifier.predict(X)) * 100:.2f}%\n")

# --- THE ULTIMATE AI TEST ---
print("--- SCENARIO 1: The False Positive Test ---")
print("User is zipping a large folder using a known program (7-Zip).")
# Fast speed (35), Max Entropy (7.9), Trusted Process (1), Header Safe (0)
scenario_1 = pd.DataFrame([[35, 7.9, 1, 0]], columns=X.columns)

if advanced_classifier.predict(scenario_1)[0] == 1:
    print("Diagnosis: 🚨 RANSOMWARE (FAIL - This is a False Positive!)")
else:
    print("Diagnosis: ✅ NORMAL BEHAVIOR (SUCCESS - AI knows it is just a ZIP file!)")


print("\n--- SCENARIO 2: The Stealth Ransomware Test ---")
print("Hacker is slowly encrypting files using an unknown script.")
# Slow speed (5), Max Entropy (7.9), Unknown Process (0), Header Destroyed (1)
scenario_2 = pd.DataFrame([[5, 7.9, 0, 1]], columns=X.columns)

if advanced_classifier.predict(scenario_2)[0] == 1:
    print("Diagnosis: 🚨 RANSOMWARE (SUCCESS - AI caught the stealth attack!)")
else:
    print("Diagnosis: ✅ NORMAL BEHAVIOR (FAIL - The hacker slipped past!)")

[*] Upgrading AI Brain with Advanced Kernel Features...
[*] Training the Advanced Scikit-Learn Classifier...
[+] Advanced AI Model Training Complete! Accuracy: 100.00%

--- SCENARIO 1: The False Positive Test ---
User is zipping a large folder using a known program (7-Zip).
Diagnosis: ✅ NORMAL BEHAVIOR (SUCCESS - AI knows it is just a ZIP file!)

--- SCENARIO 2: The Stealth Ransomware Test ---
Hacker is slowly encrypting files using an unknown script.
Diagnosis: ✅ NORMAL BEHAVIOR (FAIL - The hacker slipped past!)


In [9]:
!pip install psutil

In [10]:
import subprocess
import psutil
import time
import os
from IPython.display import clear_output, display, HTML

def launch_simulated_malware():
    """
    Launches a harmless background program that just sleeps,
    but we will pretend it is our active ransomware encrypting files.
    """
    print("\n[💀] MALICIOUS ACTOR: Launching stealth encryption payload...")
    # We launch a separate, independent Python process in the background
    malware = subprocess.Popen(["python3", "-c", "import time; time.sleep(300)"])
    return malware.pid

def neutralize_threat(pid):
    """
    The Weaponized Payload of our Daemon.
    It hunts down the Process ID and kills it at the OS level.
    """
    print(f"\n[⚡] DAEMON: Initiating automated strike against PID {pid}...")
    time.sleep(1) # Dramatic pause for the simulation

    try:
        target_process = psutil.Process(pid)

        # 1. Suspend the process immediately to stop encryption in its tracks
        target_process.suspend()
        print(f"[*] DAEMON: Process {pid} suspended. I/O locked.")

        # 2. Terminate the process completely
        target_process.terminate()

        # 3. Wait for the OS to confirm the kill
        target_process.wait(timeout=3)

        display(HTML("<h2 style='color:green; padding:10px;'>✅ THREAT NEUTRALIZED</h2>"))
        print(f"[+] ACTION SUCCESS: Malicious process ({pid}) has been destroyed.")
        print("[+] NETWORK ISOLATION: (Simulated) Host machine disconnected from network.")

    except psutil.NoSuchProcess:
        print(f"[!] Error: Process {pid} does not exist or already died.")
    except psutil.AccessDenied:
        print(f"[!] Error: Access Denied. EntropyShield requires Administrator privileges!")

# --- THE FINAL SHOWDOWN ---

# 1. The hacker launches their attack
malware_pid = launch_simulated_malware()
print(f"[*] SYSTEM KERNEL: Unknown process spawned with PID {malware_pid}")
time.sleep(2)

# 2. The AI detects it (Simulated Phase 3 output)
print("\n[*] EntropyShield AI is analyzing the behavior...")
time.sleep(2)
display(HTML("<h2 style='color:red; background-color:black; padding:10px;'>🚨 CRITICAL ALERT: RANSOMWARE DETECTED 🚨</h2>"))

# 3. The automated response takes over
neutralize_threat(malware_pid)


[💀] MALICIOUS ACTOR: Launching stealth encryption payload...
[*] SYSTEM KERNEL: Unknown process spawned with PID 14518

[*] EntropyShield AI is analyzing the behavior...



[⚡] DAEMON: Initiating automated strike against PID 14518...
[*] DAEMON: Process 14518 suspended. I/O locked.


[+] ACTION SUCCESS: Malicious process (14518) has been destroyed.
[+] NETWORK ISOLATION: (Simulated) Host machine disconnected from network.
